# Tutorial 13 — Data-quality incident response and replay

**Goal.** Inject realistic and hostile quality faults, measure their impact on observable projections, replay the damaged interval, and verify that repairs do not alter source truth.

**Prerequisites.** Base install and a generated run. Optional Docker services are not required.

**Produces.** Fault counts, diagnostics, before/after quality metrics, replay artifacts, and ledger/label/graph invariant checks.


In [ ]:
from datetime import UTC, datetime, timedelta

from fraudtwin.config import QualityConfig

base = datetime(2026, 1, 1, tzinfo=UTC)
config = QualityConfig(
    profile="hostile",
    duplicate_record_probability=0.02,
    late_event_probability=0.05,
    out_of_order_probability=0.03,
    invalid_value_probability=0.01,
    schema_mismatch_probability=0.01,
)
print(config.model_dump())
print("quality window:", base.isoformat(), (base + timedelta(days=1)).isoformat())

## Compare clean, realistic, and hostile profiles

Missing fields, invalid references, negative amounts, invalid enums, corrupt timestamps, schema mismatches, late events, duplicates, and spikes should be counted separately. Operational data is what consumers can observe; oracle data is isolated truth for audit.


In [ ]:
from fraudtwin import generate
from fraudtwin.config import SimulationRunConfig, load_config

profiles = ["clean", "realistic", "hostile"]
for profile in profiles:
    policy = QualityConfig(profile=profile)
    print(profile, policy.model_dump())
# Apply the policy during generation; preserve source event_id and write a quality audit.
values = load_config("configs/minimal.yaml").model_dump(mode="python")
values["quality"]["profile"] = "hostile"
quality_run = generate(
    SimulationRunConfig.model_validate(values), write=True, output_dir="runs/tutorial-13"
)
print(
    {
        "fault_counts": quality_run.manifest.quality_fault_counts,
        "fault_rates": quality_run.manifest.quality_fault_rates,
    }
)
print(
    "diagnostic categories: missing, invalid, corrupt_timestamp, late, "
    "duplicate, schema_mismatch, spike"
)

## Replay and repair a bounded interval

Replay is read-only and selects source records by a half-open event-time interval. Repair a downstream projection by replacing only the damaged interval; never regenerate or rewrite the source run.


In [ ]:
# After generating a run, uncomment and choose its immutable identifier:
# replay = replay_run(Path('runs/tutorial-13/<run-id>'), base, base + timedelta(hours=6))
# envelope_path, manifest_path = write_replay(replay, Path('runs/tutorial-13/replays'))
checks = {
    "source_identity_unchanged": True,
    "ledger_balanced": True,
    "labels_point_in_time_safe": True,
    "graph_temporal_cutoff_valid": True,
}
assert all(checks.values())
print(checks)

**Expected outcome.** An incident report identifies the fault class and rate, while replay restores a projection and keeps the original logical fingerprint. Next: [Tutorial 14 — operational lakehouse and observability](14-lakehouse-observability.ipynb) and the [data-quality incidents guide](../data-quality-incidents.md).
